# Alpha sweep — exploratory analysis (CLIPSeg)

Companion to `analyze_alpha.py`. That script is the **reproducible report**; this notebook
is for **digging**. Point `RUN_DIR` at a finished run and work down.

**Kernel:** pick the `tasa` env. If it isn't listed:
`/home/sschaud5/.conda/envs/tasa/bin/python -m pip install ipykernel pandas matplotlib`
then reload the VS Code window.

In [ ]:
RUN_DIR = "/home/sschaud5/workspace/OVS/results/alpha_clipseg_CHANGEME"   # <-- edit me

import glob, os, json
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)

detail = sorted(glob.glob(os.path.join(RUN_DIR, "*_detail.csv")))[0]
df = pd.read_csv(detail)
df["weighted"] = df["weighted"].astype(str) == "True"

print(detail)
print(f"{len(df):,} rows | {df.category.nunique()} categories | "
      f"{df.img_ref.nunique():,} images | grids={sorted(df.grid.unique())}")
df.head()

### Provenance — what produced this run

In [ ]:
meta = json.load(open(os.path.join(RUN_DIR, "run_meta.json")))
for k in ["started_utc", "wall_seconds", "status", "n_detail_rows", "host"]:
    print(f"{k:16} {meta.get(k)}")
print(f"{'commit':16} {meta.get('git',{}).get('commit','?')[:12]} "
      f"(dirty={meta.get('git',{}).get('dirty')})")
print(f"{'args':16} {meta.get('args')}")
print(f"{'packages':16} {meta.get('packages')}")

## 1. Headline — does blending beat the baseline?

`alpha=0` is the no-blend baseline. Everything else is `(1-a)*query + a*centroid`.

In [ ]:
def curve(sub):
    return sub.groupby("alpha").iou.mean().sort_index()

baseline = curve(df[df.grid == "coarse"]).get(0.0)
rows = []
for grid in sorted(df.grid.unique()):
    for w in [False, True]:
        s = curve(df[(df.grid == grid) & (df.weighted == w)])
        if s.empty: continue
        rows.append(dict(grid=grid, centroid="weighted" if w else "unweighted",
                         best_alpha=s.idxmax(), iou_at_best=round(s.max(), 4),
                         baseline=round(baseline, 4),
                         abs_gain=round(s.max()-baseline, 4),
                         rel_gain_pct=round(100*(s.max()-baseline)/baseline, 1)))
head = pd.DataFrame(rows)
best = head.loc[head.iou_at_best.idxmax()]
WEIGHTED = best.centroid == "weighted"
ALPHA    = best.best_alpha
display(head)
print(f"==> USE alpha={ALPHA:.2f}, weighted={WEIGHTED}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, grid in zip(axes, ["coarse", "fine"]):
    d = df[df.grid == grid]
    if d.empty: continue
    for w in [False, True]:
        s = curve(d[d.weighted == w])
        if s.empty: continue
        ax.plot(s.index, s.values, marker="o", ms=3,
                label=("weighted" if w else "unweighted"))
        ax.scatter([s.idxmax()], [s.max()], zorder=5)
    if grid == "coarse":
        ax.axhline(baseline, ls="--", c="gray", lw=1)
    ax.set_title(f"{grid} grid"); ax.set_xlabel("alpha"); ax.grid(alpha=.3); ax.legend()
axes[0].set_ylabel("mean IoU"); plt.tight_layout()

## 2. Plateau — how sensitive is this to the exact alpha?

A wide band = robust method. A narrow spike = fragile, look harder.

In [ ]:
g = df[(df.grid == ("fine" if (df.grid=="fine").any() else "coarse")) & (df.weighted == WEIGHTED)]
s = curve(g)  
flat = s[s >= s.max()*0.99]
print(f"peak      {s.max():.4f} at alpha={s.idxmax():.2f}")
print(f"within 1% alpha {flat.index.min():.2f} .. {flat.index.max():.2f}  "
      f"({len(flat)}/{len(s)} grid points, spread {flat.max()-flat.min():.4f})")

## 3. Per variant — **the core result**

The grand mean hides the thesis. Expect `hypo`/`hyper` (rare, abstract words) to gain a lot
and `orig` (the canonical word the model already knows) to lose a little.

In [ ]:
sub = df[(df.grid == "coarse") & (df.weighted == WEIGHTED)]
piv = sub.pivot_table(index="alpha", columns="variant", values="iou", aggfunc="mean")
piv = piv[[c for c in ["orig","syn","hypo","hyper"] if c in piv.columns]]
display(piv.round(4))

summary = pd.DataFrame({
    "iou_at_0":    piv.loc[0.0].round(4),
    "best_alpha":  piv.idxmax(),
    "iou_at_best": piv.max().round(4),
    "gain":        (piv.max() - piv.loc[0.0]).round(4),
    "n_rows":      sub.groupby("variant").size(),
})
display(summary)

In [ ]:
ax = piv.plot(marker="o", ms=4, figsize=(8,5), grid=True)
ax.axvline(ALPHA, ls="--", c="k", lw=1, alpha=.5)
ax.set_xlabel("alpha"); ax.set_ylabel("mean IoU")
ax.set_title(f"per-variant ({best.centroid} centroid)");

### Which actual words are behind each variant?

In [ ]:
words = (df[df.variant != "orig"]
         .groupby(["variant","category"]).variant_word.first()
         .unstack(0))
display(words.head(40))

## 4. Per category — where does it help, where does it hurt?

In [ ]:
cw   = df[(df.grid == "coarse") & (df.weighted == WEIGHTED)]
cpiv = cw.pivot_table(index="category", columns="alpha", values="iou", aggfunc="mean")
a    = min(cpiv.columns, key=lambda x: abs(x - ALPHA))
gain = (cpiv[a] - cpiv[0.0]).sort_values()

print(f"helped {(gain>0).sum()}/{len(gain)} categories | median {gain.median():+.4f}")
fig, ax = plt.subplots(figsize=(8, .3*len(gain)+2))
ax.barh(gain.index, gain.values,
        color=["#c0392b" if v<0 else "#27ae60" for v in gain.values])
ax.axvline(0, c="k", lw=1); ax.set_xlabel(f"IoU(a={a:.2f}) - IoU(a=0)")
plt.tight_layout()

## 5. Free-form exploration

Recipes below — edit and rerun. `df` columns:
`grid, category, variant, variant_word, weighted, img_ref, alpha, iou`

**a) Per-category × per-variant gain — find where hypernyms are rescued most**

In [ ]:
h = df[(df.grid=="coarse") & (df.weighted==WEIGHTED) & (df.variant=="hyper")]
hp = h.pivot_table(index="category", columns="alpha", values="iou", aggfunc="mean")
(hp[a] - hp[0.0]).sort_values(ascending=False).head(15)

**b) Does weighted actually differ from unweighted anywhere?**

In [ ]:
w = df[df.grid=="coarse"].pivot_table(index=["category","alpha"],
                                      columns="weighted", values="iou", aggfunc="mean")
w["diff"] = w[True] - w[False]
w["diff"].abs().sort_values(ascending=False).head(15)

**c) Per-image spread at the chosen alpha — is the mean hiding bimodality?**

In [ ]:
d = df[(df.grid=="coarse") & (df.weighted==WEIGHTED) & (df.alpha==a)]
d.groupby("variant").iou.describe().round(3)

**d) Best alpha computed per category instead of globally**

In [ ]:
pc = (df[(df.grid=="fine") & (df.weighted==WEIGHTED)]
      .pivot_table(index="category", columns="alpha", values="iou", aggfunc="mean"))
pd.DataFrame({"best_alpha": pc.idxmax(axis=1), "iou": pc.max(axis=1).round(4)}) \
  .sort_values("best_alpha")

**e) Scratch**